In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.resampling import (
    load_time_indexed_csv,
    ensure_clean_datetime_index,
    infer_default_rule_groups,
    build_resample_rule_map,
    convert_and_save_resolution,
    convert_multiple_resolutions,
    summarize_resampled_dataframe,
)

In [2]:
INPUT_PATH = r"C:\Data_analysis\Thesis\Data\02_Preprocessing\LF_data_droped.csv"
df_5min = load_time_indexed_csv(INPUT_PATH)
df_5min = ensure_clean_datetime_index(df_5min)

print(df_5min.shape)
df_5min.head()

(33696, 9)


,BA_Soc,BU_TotActPwr_Academy,BA_TotActPwr_BESS_AC_Panel1,BA_TotActPwr_BESS_AC_Panel2,BU_TotActPwr_SDB_EL_Substation,BU_TotActPwr_Tech_Room,PV_WS_AirTemp,PV_WS_Radiation,PV_WS_RelHum
Time,,,,,,,,,
2025-10-15 00:00:00,83.4,4.148,1.630,0.419,NaN,3.614,129.0,-4.6,83.5
2025-10-15 00:05:00,83.4,3.631,1.632,0.421,NaN,3.608,129.0,-4.3,83.8
2025-10-15 00:10:00,83.3,3.572,1.633,0.420,NaN,3.646,127.0,-4.7,84.1
2025-10-15 00:15:00,83.2,3.663,1.636,0.418,NaN,3.601,127.0,-4.5,84.3
2025-10-15 00:20:00,83.1,3.610,1.634,0.426,NaN,3.646,126.0,-4.5,85.0


In [3]:
rule_groups = infer_default_rule_groups(df_5min)

mean_cols = rule_groups["mean_cols"]
last_cols = rule_groups["last_cols"]
max_cols = rule_groups["max_cols"]
min_cols = rule_groups["min_cols"]
sum_cols = rule_groups["sum_cols"]

print("Mean cols:", len(mean_cols))
print("Last cols:", len(last_cols))

Mean cols: 9
Last cols: 0


In [4]:
manual_last_cols = [
    "PV_Unitstate",
    "BU_Unitstate",
    "BA_Unitstate",
]
manual_last_cols = [c for c in manual_last_cols if c in df_5min.columns]

# remove from mean if present
mean_cols = [c for c in mean_cols if c not in manual_last_cols]
last_cols = sorted(set(last_cols + manual_last_cols))

In [5]:
agg_map = build_resample_rule_map(
    df_5min,
    mean_cols=mean_cols,
    last_cols=last_cols,
    max_cols=max_cols,
    min_cols=min_cols,
    sum_cols=sum_cols,
    default_numeric_rule="mean",
)

In [6]:
OUT_CSV = r"C:\Data_analysis\Thesis\Data\02_Preprocessing\LF_data_droped_15m.csv"

df_15min, saved = convert_and_save_resolution(
    df_5min,
    target_freq="15min",
    agg_map=agg_map,
    csv_path=OUT_CSV,
    label="left",
    closed="left",
    drop_all_nan_rows=False,
)

print(saved)
print(df_15min.shape)
df_15min.head()

{'csv': WindowsPath('C:/Data_analysis/Thesis/Data/02_Preprocessing/LF_data_droped_15m.csv')}
(11232, 9)


,BA_Soc,BU_TotActPwr_Academy,BA_TotActPwr_BESS_AC_Panel1,BA_TotActPwr_BESS_AC_Panel2,BU_TotActPwr_SDB_EL_Substation,BU_TotActPwr_Tech_Room,PV_WS_AirTemp,PV_WS_Radiation,PV_WS_RelHum
Time,,,,,,,,,
2025-10-15 00:00:00,83.366667,3.783667,1.631667,0.420000,NaN,3.622667,128.333333,-4.533333,83.800000
2025-10-15 00:15:00,83.100000,3.646667,1.635333,0.420000,NaN,3.620667,127.000000,-4.366667,84.800000
2025-10-15 00:30:00,82.800000,3.630333,1.634333,0.418000,NaN,3.685667,126.333333,-4.433333,85.400000
2025-10-15 00:45:00,82.566667,4.424667,1.929333,0.417000,NaN,3.683667,125.000000,-4.700000,85.700000
2025-10-15 01:00:00,82.266667,3.567000,1.928000,0.513667,NaN,3.598333,123.333333,-4.500000,86.433333


In [7]:
summary = summarize_resampled_dataframe(
    df_5min,
    df_15min,
    original_freq="5min",
    target_freq="15min",
)

print(summary.to_string(index=False))

       metric               value
  rows_before               33696
   rows_after               11232
  cols_before                   9
   cols_after                   9
 start_before 2025-10-15 00:00:00
   end_before 2026-02-08 23:55:00
  start_after 2025-10-15 00:00:00
    end_after 2026-02-08 23:45:00
original_freq                5min
  target_freq               15min


In [8]:
OUTPUT_DIR = r"C:\Data_analysis\Thesis\Data\02_Preprocessing\resampled"

results = convert_multiple_resolutions(
    df_5min,
    target_freqs=["15min", "30min"],
    agg_map=agg_map,
    output_dir=OUTPUT_DIR,
    file_stem="LF_data_droped",
    save_csv=True,
    label="left",
    closed="left",
    drop_all_nan_rows=False,
)

for freq, obj in results.items():
    print(freq, obj["df"].shape)
    if "csv" in obj:
        print(" csv:", obj["csv"])


15min (11232, 9)
 csv: C:\Data_analysis\Thesis\Data\02_Preprocessing\resampled\LF_data_droped_15m.csv
30min (5616, 9)
 csv: C:\Data_analysis\Thesis\Data\02_Preprocessing\resampled\LF_data_droped_30m.csv
